# Ciliary Cluster Analysis - Synthetic Demo Pipeline

This notebook provides a **fully self-contained synthetic demonstration** of the ciliary gene cluster analysis pipeline. All data is artificially generated with engineered structure to showcase the analytical workflow without exposing sensitive or proprietary information.

## Overview

This demo demonstrates:
- **Structured synthetic gene cluster generation** (no white noise)
- **Similarity-based characterization** of gene profiles
- **Hierarchical clustering** and cluster assignment
- **Cluster coherence analysis** (intra-cluster similarity metrics)
- **Publication-ready visualizations** (heatmaps, barplots)
- **Systematic output generation** (CSV tables, PNG figures)

---

## Workflow Summary

1. **Data Generation**: Create synthetic gene × species phylogenetic profiles with correlated cluster structure
2. **Similarity Computation**: Calculate Pearson correlation between gene profiles
3. **Clustering**: Apply hierarchical clustering to group similar genes
4. **Coherence Analysis**: Quantify internal cluster quality via intra-cluster similarity
5. **Visualization**: Generate heatmaps and barplots
6. **Output Export**: Save all results to `results/` directory

---

**Note**: All gene names, species names, and data values are synthetic. This demo mirrors the conceptual workflow of the real pipeline but uses only artificial data.


## Section A: Setup & Imports

Load required libraries and ensure output directory exists.


In [ ]:
# Standard library imports
import os
from pathlib import Path

# Scientific computing
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for publication-ready plots
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

# Create results directory if it doesn't exist
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

print("✓ Imports loaded successfully")
print(f"✓ Results directory ready: {results_dir.absolute()}")


## Section B: Structured Synthetic Data Generation

Generate synthetic gene × species phylogenetic profiles with engineered cluster structure. Each cluster has:
- A base evolutionary profile
- Genes with correlated deviations from the base profile
- Controlled noise levels to maintain cluster coherence


In [ ]:
def generate_synthetic_cilia_clusters(
    n_clusters=3,
    genes_per_cluster=20,
    n_species=50,
    intra_cluster_corr=0.6,
    noise_level=0.2,
    random_state=42,
):
    """
    Generates structured synthetic ciliary-like clusters for demonstration.

    Parameters
    ----------
    n_clusters : int
        Number of distinct gene clusters to generate
    genes_per_cluster : int
        Number of genes per cluster
    n_species : int
        Number of species (columns) in the phylogenetic profile matrix
    intra_cluster_corr : float
        Target correlation within clusters (controls cluster coherence)
    noise_level : float
        Standard deviation of noise added to profiles
    random_state : int
        Random seed for reproducibility

    Returns
    -------
    matrix : pd.DataFrame
        Gene × species synthetic profile matrix with correlated structure.
        Index: gene names (e.g., "Cluster1_Gene1")
        Columns: species names (e.g., "Species_001")
    cluster_labels : pd.Series
        Mapping from gene name to true cluster ID (1-indexed)
    """
    np.random.seed(random_state)

    total_genes = n_clusters * genes_per_cluster
    gene_names = []
    true_labels = []

    # Initialize the full matrix
    matrix_data = np.zeros((total_genes, n_species))

    gene_idx = 0
    for cluster_id in range(1, n_clusters + 1):
        # Generate a base profile for this cluster
        # Each cluster has a distinct evolutionary signature
        base_profile = np.random.randn(n_species) * 2.0
        base_profile = base_profile - base_profile.mean()  # Center

        # Add some species-specific patterns (some species have higher/lower presence)
        species_pattern = np.sin(np.linspace(0, 4 * np.pi, n_species)) * 0.5
        base_profile = base_profile + species_pattern

        for gene_num in range(1, genes_per_cluster + 1):
            # Create gene profile by adding correlated deviation from base
            # Use a weighted combination to achieve target correlation
            correlation_weight = np.sqrt(intra_cluster_corr)
            noise_weight = np.sqrt(1 - intra_cluster_corr)

            # Correlated component (shared with cluster)
            correlated_component = correlation_weight * base_profile

            # Independent noise component
            independent_noise = noise_weight * np.random.randn(n_species) * noise_level

            # Additional random noise
            random_noise = np.random.randn(n_species) * noise_level

            # Combine to form gene profile
            gene_profile = correlated_component + independent_noise + random_noise

            # Store
            matrix_data[gene_idx, :] = gene_profile
            gene_names.append(f"Cluster{cluster_id}_Gene{gene_num:02d}")
            true_labels.append(cluster_id)
            gene_idx += 1

    # Create species names
    species_names = [f"Species_{i+1:03d}" for i in range(n_species)]

    # Build DataFrame
    matrix = pd.DataFrame(
        matrix_data,
        index=gene_names,
        columns=species_names
    )

    # Create true labels Series
    cluster_labels = pd.Series(true_labels, index=gene_names, name="true_cluster")

    return matrix, cluster_labels


# Generate synthetic data
print("Generating synthetic ciliary gene clusters...")
demo_matrix, true_labels = generate_synthetic_cilia_clusters(
    n_clusters=4,
    genes_per_cluster=25,
    n_species=60,
    intra_cluster_corr=0.65,
    noise_level=0.15,
    random_state=42
)

print(f"✓ Generated matrix shape: {demo_matrix.shape}")
print(f"✓ Number of clusters: {true_labels.nunique()}")
print(f"✓ Genes per cluster: {true_labels.value_counts().iloc[0]}")
print(f"\nFirst few genes:")
print(demo_matrix.head())
print(f"\nTrue cluster distribution:")
print(true_labels.value_counts().sort_index())


## Section C: Similarity Computation

Compute pairwise similarity between genes using Pearson correlation. This measures how similar the evolutionary profiles are across species.


In [ ]:
def compute_similarity_matrix(matrix, metric="pearson"):
    """
    Compute pairwise similarity matrix between genes.

    Parameters
    ----------
    matrix : pd.DataFrame
        Gene × species matrix
    metric : str
        Correlation method ('pearson' or 'spearman')

    Returns
    -------
    similarity_matrix : pd.DataFrame
        Gene × gene similarity matrix (correlation coefficients)
    """
    # Compute correlation between genes (transpose so genes are columns)
    # Then transpose back so genes are both rows and columns
    similarity = matrix.T.corr(method=metric)

    # Ensure diagonal is exactly 1.0 (self-correlation)
    np.fill_diagonal(similarity.values, 1.0)

    return similarity


# Compute similarity matrix
print("Computing gene-gene similarity matrix...")
similarity_matrix = compute_similarity_matrix(demo_matrix, metric="pearson")

print(f"✓ Similarity matrix shape: {similarity_matrix.shape}")
print(f"✓ Similarity range: [{similarity_matrix.values.min():.3f}, {similarity_matrix.values.max():.3f}]")
print(f"✓ Mean similarity: {similarity_matrix.values.mean():.3f}")
print(f"\nSimilarity matrix preview (first 5×5):")
print(similarity_matrix.iloc[:5, :5].round(3))


## Section D: Clustering & Coherence Analysis

Apply hierarchical clustering to group genes based on similarity, then compute cluster coherence metrics to assess internal cluster quality.


In [ ]:
def run_hierarchical_clustering(similarity_matrix, n_clusters=None, linkage_method="average"):
    """
    Perform hierarchical clustering on similarity matrix.

    Parameters
    ----------
    similarity_matrix : pd.DataFrame
        Gene × gene similarity matrix
    n_clusters : int, optional
        Number of clusters to form. If None, will use optimal number based on silhouette score.
    linkage_method : str
        Linkage method ('average', 'ward', 'complete', 'single')

    Returns
    -------
    cluster_labels : pd.Series
        Inferred cluster assignments (1-indexed)
    linkage_matrix : np.ndarray
        Hierarchical clustering linkage matrix
    """
    # Convert similarity to distance (1 - similarity)
    # Note: For 'ward' linkage, we need squared Euclidean distance
    if linkage_method == "ward":
        # Ward requires squared Euclidean distance
        # Convert similarity to distance, then square
        distances = 1 - similarity_matrix.values
        distances = np.clip(distances, 0, None)  # Ensure non-negative
        distance_matrix = squareform(distances, checks=False)
    else:
        # For other linkage methods, use 1 - similarity as distance
        distances = 1 - similarity_matrix.values
        np.fill_diagonal(distances, 0)  # Self-distance is 0
        distance_matrix = squareform(distances, checks=False)

    # Perform hierarchical clustering
    linkage_matrix = linkage(distance_matrix, method=linkage_method)

    # Determine optimal number of clusters if not specified
    if n_clusters is None:
        # Test range of cluster numbers and pick best silhouette score
        best_score = -1
        best_n = 2
        for n in range(2, min(10, len(similarity_matrix) // 5)):
            labels = fcluster(linkage_matrix, n, criterion='maxclust')
            if len(set(labels)) > 1:  # Need at least 2 clusters for silhouette
                score = silhouette_score(1 - similarity_matrix.values, labels, metric='precomputed')
                if score > best_score:
                    best_score = score
                    best_n = n
        n_clusters = best_n
        print(f"  Optimal number of clusters (by silhouette): {n_clusters}")

    # Extract cluster assignments
    inferred_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

    # Create Series with gene names as index
    cluster_labels = pd.Series(
        inferred_labels,
        index=similarity_matrix.index,
        name="inferred_cluster"
    )

    return cluster_labels, linkage_matrix


# Determine number of clusters from true labels
n_clusters = true_labels.nunique()
print(f"Running hierarchical clustering with {n_clusters} clusters...")
inferred_labels, linkage_matrix = run_hierarchical_clustering(
    similarity_matrix,
    n_clusters=n_clusters,
    linkage_method="average"
)

print(f"✓ Clustering complete")
print(f"✓ Inferred {inferred_labels.nunique()} clusters")
print(f"\nCluster size distribution:")
print(inferred_labels.value_counts().sort_index())


In [ ]:
def compute_cluster_coherence(similarity_matrix, cluster_labels):
    """
    Compute cluster coherence metrics (intra-cluster similarity statistics).

    Parameters
    ----------
    similarity_matrix : pd.DataFrame
        Gene × gene similarity matrix
    cluster_labels : pd.Series
        Cluster assignments for each gene

    Returns
    -------
    cluster_stats : pd.DataFrame
        Statistics per cluster (mean intra-cluster similarity, size, etc.)
    """
    results = []

    for cluster_id in sorted(cluster_labels.unique()):
        # Get genes in this cluster
        cluster_genes = cluster_labels[cluster_labels == cluster_id].index.tolist()
        n_genes = len(cluster_genes)

        if n_genes < 2:
            # Can't compute intra-cluster similarity with < 2 genes
            results.append({
                "cluster_id": cluster_id,
                "n_genes": n_genes,
                "mean_intra_similarity": np.nan,
                "std_intra_similarity": np.nan,
                "min_intra_similarity": np.nan,
                "max_intra_similarity": np.nan,
            })
            continue

        # Extract submatrix for this cluster
        cluster_sim = similarity_matrix.loc[cluster_genes, cluster_genes]

        # Get upper triangle (excluding diagonal) for intra-cluster similarities
        mask = np.triu(np.ones_like(cluster_sim.values, dtype=bool), k=1)
        intra_similarities = cluster_sim.values[mask]

        results.append({
            "cluster_id": cluster_id,
            "n_genes": n_genes,
            "mean_intra_similarity": float(np.mean(intra_similarities)),
            "std_intra_similarity": float(np.std(intra_similarities)),
            "min_intra_similarity": float(np.min(intra_similarities)),
            "max_intra_similarity": float(np.max(intra_similarities)),
        })

    return pd.DataFrame(results)


# Compute cluster coherence statistics
print("Computing cluster coherence metrics...")
cluster_stats = compute_cluster_coherence(similarity_matrix, inferred_labels)

print("✓ Cluster coherence analysis complete")
print(f"\nCluster statistics:")
print(cluster_stats.round(3))

# Compare inferred vs true clusters (if we have true labels)
if true_labels is not None:
    # Create comparison DataFrame
    comparison = pd.DataFrame({
        "gene": similarity_matrix.index,
        "true_cluster": true_labels,
        "inferred_cluster": inferred_labels
    })

    # Compute accuracy (adjusted Rand index would be better, but simple accuracy for demo)
    # For each true cluster, find the most common inferred cluster
    from scipy.stats import mode
    accuracy_data = []
    for true_clust in sorted(true_labels.unique()):
        true_genes = comparison[comparison["true_cluster"] == true_clust]
        if len(true_genes) > 0:
            inferred_mode = mode(true_genes["inferred_cluster"], keepdims=True).mode[0]
            n_correct = (true_genes["inferred_cluster"] == inferred_mode).sum()
            accuracy_data.append({
                "true_cluster": true_clust,
                "most_common_inferred": inferred_mode,
                "n_genes": len(true_genes),
                "n_correct": n_correct,
                "accuracy": n_correct / len(true_genes)
            })

    accuracy_df = pd.DataFrame(accuracy_data)
    print(f"\n✓ Clustering accuracy (true vs inferred):")
    print(accuracy_df.round(3))


## Section E: Saving Outputs

Export all generated data and analysis results to CSV files in the `results/` directory.


In [ ]:
# Save all outputs to results directory
print("Saving outputs to results/ directory...")

# 1. Gene × species matrix
matrix_path = results_dir / "demo_cilia_matrix.csv"
demo_matrix.to_csv(matrix_path)
print(f"✓ Saved: {matrix_path.name}")

# 2. Similarity matrix
similarity_path = results_dir / "demo_similarity_matrix.csv"
similarity_matrix.to_csv(similarity_path)
print(f"✓ Saved: {similarity_path.name}")

# 3. Cluster assignments
assignments_path = results_dir / "demo_cluster_assignments.csv"
assignments_df = pd.DataFrame({
    "gene": inferred_labels.index,
    "inferred_cluster": inferred_labels.values,
    "true_cluster": true_labels.values if true_labels is not None else None
})
assignments_df.to_csv(assignments_path, index=False)
print(f"✓ Saved: {assignments_path.name}")

# 4. Cluster statistics
stats_path = results_dir / "demo_cluster_stats.csv"
cluster_stats.to_csv(stats_path, index=False)
print(f"✓ Saved: {stats_path.name}")

print(f"\n✓ All outputs saved to: {results_dir.absolute()}")


## Section F: Visualizations

Generate publication-ready visualizations: similarity heatmap and cluster coherence barplot.


In [ ]:
def plot_similarity_heatmap(similarity_matrix, cluster_labels, outpath, figsize=(12, 10)):
    """
    Plot heatmap of similarity matrix with cluster annotations.

    Parameters
    ----------
    similarity_matrix : pd.DataFrame
        Gene × gene similarity matrix
    cluster_labels : pd.Series
        Cluster assignments for ordering
    outpath : Path
        Output file path
    figsize : tuple
        Figure size in inches
    """
    # Reorder genes by cluster for better visualization
    cluster_order = cluster_labels.sort_values()
    ordered_sim = similarity_matrix.loc[cluster_order.index, cluster_order.index]

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Plot heatmap
    sns.heatmap(
        ordered_sim,
        cmap="viridis",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        cbar_kws={"label": "Pearson Correlation", "shrink": 0.8},
        xticklabels=False,
        yticklabels=False,
        ax=ax
    )

    # Add cluster boundaries
    cluster_boundaries = []
    current_cluster = None
    for idx, (gene, cluster) in enumerate(cluster_order.items()):
        if cluster != current_cluster:
            if current_cluster is not None:
                cluster_boundaries.append(idx)
            current_cluster = cluster

    # Draw vertical and horizontal lines at cluster boundaries
    for boundary in cluster_boundaries:
        ax.axhline(y=boundary, color='white', linewidth=2, alpha=0.7)
        ax.axvline(x=boundary, color='white', linewidth=2, alpha=0.7)

    ax.set_title("Gene-Gene Similarity Heatmap\n(Ordered by Inferred Clusters)",
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel("Genes", fontsize=12)
    ax.set_ylabel("Genes", fontsize=12)

    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved heatmap: {outpath.name}")


# Generate similarity heatmap
heatmap_path = results_dir / "demo_similarity_heatmap.png"
plot_similarity_heatmap(similarity_matrix, inferred_labels, heatmap_path, figsize=(12, 10))


In [ ]:
def plot_cluster_coherence_barplot(cluster_stats, outpath, figsize=(10, 6)):
    """
    Plot barplot of cluster coherence (mean intra-cluster similarity).

    Parameters
    ----------
    cluster_stats : pd.DataFrame
        Cluster statistics DataFrame
    outpath : Path
        Output file path
    figsize : tuple
        Figure size in inches
    """
    fig, ax = plt.subplots(figsize=figsize)

    # Sort by cluster ID
    stats_sorted = cluster_stats.sort_values("cluster_id")

    # Create barplot
    bars = ax.bar(
        stats_sorted["cluster_id"].astype(str),
        stats_sorted["mean_intra_similarity"],
        color=sns.color_palette("husl", len(stats_sorted)),
        edgecolor='black',
        linewidth=1.5,
        alpha=0.8
    )

    # Add error bars (std)
    ax.errorbar(
        stats_sorted["cluster_id"].astype(str),
        stats_sorted["mean_intra_similarity"],
        yerr=stats_sorted["std_intra_similarity"],
        fmt='none',
        color='black',
        capsize=5,
        capthick=1.5
    )

    # Add value labels on bars
    for i, (idx, row) in enumerate(stats_sorted.iterrows()):
        height = row["mean_intra_similarity"]
        ax.text(
            i,
            height + row["std_intra_similarity"] + 0.02,
            f'{height:.3f}',
            ha='center',
            va='bottom',
            fontsize=9,
            fontweight='bold'
        )
        # Add gene count label
        ax.text(
            i,
            -0.05,
            f'n={int(row["n_genes"])}',
            ha='center',
            va='top',
            fontsize=8,
            style='italic'
        )

    ax.set_xlabel("Cluster ID", fontsize=12, fontweight='bold')
    ax.set_ylabel("Mean Intra-Cluster Similarity", fontsize=12, fontweight='bold')
    ax.set_title("Cluster Coherence Analysis\n(Mean Intra-Cluster Pearson Correlation)",
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_ylim(bottom=-0.1, top=1.0)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.axhline(y=0, color='black', linewidth=0.8)

    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved barplot: {outpath.name}")


# Generate coherence barplot
barplot_path = results_dir / "demo_cluster_coherence.png"
plot_cluster_coherence_barplot(cluster_stats, barplot_path, figsize=(10, 6))


In [ ]:
# Display the heatmap inline for immediate visualization
heatmap_path = results_dir / "demo_similarity_heatmap.png"
if heatmap_path.exists():
    from IPython.display import Image, display
    display(Image(str(heatmap_path)))


In [ ]:
# Display the coherence barplot inline
barplot_path = results_dir / "demo_cluster_coherence.png"
if barplot_path.exists():
    from IPython.display import Image, display
    display(Image(str(barplot_path)))


## Section G: Final Summary

### What This Demo Accomplishes

This synthetic demonstration showcases a complete computational pipeline for analyzing ciliary gene clusters:

1. **Structured Data Generation**: Creates realistic gene × species phylogenetic profiles with engineered cluster structure, where genes within clusters share correlated evolutionary patterns.

2. **Similarity-Based Characterization**: Computes pairwise gene-gene similarity using Pearson correlation, revealing which genes have similar evolutionary profiles across species.

3. **Unsupervised Clustering**: Applies hierarchical clustering to automatically group genes based on their similarity patterns, without prior knowledge of cluster assignments.

4. **Quality Assessment**: Quantifies cluster coherence through intra-cluster similarity metrics, providing objective measures of how well genes within each cluster are related.

5. **Visual Communication**: Generates publication-ready visualizations that clearly illustrate:
   - The block-diagonal structure in similarity matrices (indicating strong cluster structure)
   - The coherence levels of each inferred cluster

### How It Relates to the Real Pipeline

This demo mirrors the conceptual workflow of the real ciliary cluster analysis pipeline:

- **Data Structure**: The synthetic gene × species matrix mimics real phylogenetic profile (NPP) matrices used in evolutionary genomics
- **Analysis Steps**: The sequence (similarity → clustering → coherence) reflects the systematic approach used to characterize precomputed clusters
- **Output Format**: CSV tables and PNG visualizations match the output structure of the production pipeline
- **Quality Metrics**: Coherence analysis provides the same type of validation used to assess real cluster quality

### Why the Data is Synthetic

All data in this demo is **fully synthetic**:
- Gene names are artificial (`Cluster1_Gene01`, etc.)
- Species names are generic (`Species_001`, etc.)
- All numerical values are generated from controlled random distributions
- No real biological data or proprietary algorithms are exposed

This ensures the demo is:
- **Safe to publish** (no sensitive information)
- **Reproducible** (deterministic with fixed random seed)
- **Educational** (clear structure demonstrates concepts)
- **Portfolio-ready** (showcases methodology without revealing proprietary details)

### Biological Relevance

While synthetic, the demo structure reflects biologically meaningful patterns:

- **Cluster Coherence**: Real ciliary gene clusters (e.g., IFT, BBS) show high intra-cluster similarity due to co-evolution
- **Modular Structure**: The block-diagonal patterns in similarity matrices mirror the modular organization of ciliary systems
- **Evolutionary Signals**: The correlation-based similarity captures the type of evolutionary relationships used to identify functional gene modules

### Next Steps

To run the full production pipeline on real data:
1. Replace synthetic data generation with real NPP matrix loading
2. Apply the same similarity computation and clustering steps
3. Integrate with functional annotation databases (CiliaCarta, GO, etc.)
4. Perform enrichment analysis and evidence integration
5. Generate publication-ready summaries and visualizations

---

**Demo Complete!** All outputs have been saved to the `results/` directory.
